# P130 — Los modelos de lenguaje sobre códecs neuronales sintetizan voz sin ejemplos previos

## 1. Título y paper

**Paper:** *Neural Codec Language Models are Zero-Shot Text to Speech Synthesizers*  
**Autoría:** Chengyi Wang, Sanyuan Chen, Yu Wu, Ziqiang Zhang, Long Zhou, Shujie Liu, y otros  
**Año y venue:** 2023 · arXiv:2301.02111  
**Nivel:** L3 · **Motor:** `vall_e`  
**Ficha completa:** [`P130_vall_e`](../../papers/foundational/P130_vall_e/README.md)

**Hito:** Convierte la síntesis de voz en modelado de lenguaje sobre códigos de audio, y clona una voz con tres segundos de muestra sin entrenar nada.

- [arXiv:2301.02111](https://arxiv.org/abs/2301.02111)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Adaptar un sintetizador a una voz nueva exigía media hora o más de grabaciones y un ajuste fino del modelo. Eso limitaba la personalización a quien tuviera estudio, y de paso actuaba como barrera práctica frente al uso indebido.
2. Ejecutar una implementación mínima de la propuesta: Tratar los códigos de un códec neuronal como un vocabulario y la síntesis como predicción del siguiente token, con la voz objetivo entrada como aviso en contexto. Sin entrenamiento por hablante: tres segundos bastan.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P122
- P10


## 4. Intuición

El enfoque anterior necesitaba media hora de grabaciones y un entrenamiento para imitar una voz. Aquí bastan **tres segundos** y ningún entrenamiento. El resultado técnico y el problema ético son el mismo hecho.


## 5. Concepto mínimo

```text
Antes  : 30 min de audio + ajuste fino del modelo
Ahora  :  3 s de audio como AVISO EN CONTEXTO, sin entrenar

TTS deja de ser síntesis y pasa a ser predicción del siguiente token
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('vall_e', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánta identidad hay en medio segundo de voz?
2. ¿Y en tres?
3. ¿Cuánto se gana pasando de tres a treinta?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('vall_e', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('vall_e', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con **0,5 s**, la identidad del hablante se recupera el **35,4 %** de las veces. Con **3 s**, el **92,1 %**. Con **30 s**, el **100 %** — apenas **7,9 puntos** más. La información de identidad se satura muy pronto.


## 10. Comentario pedagógico

Esa saturación es exactamente lo que hace el problema irreversible. Tres segundos de la voz de cualquiera están en cualquier vídeo público, en un mensaje de voz, en una llamada. No hay consentimiento que se haya pedido ni forma de retirarlo. La contramedida —detectar voz sintética— envejece con cada generación de sintetizadores.


## 11. Error o anti-patrón deliberado

Anti-patrón: tratar la clonación de voz como un problema de calidad de audio.


In [ ]:
print('Un clon imperfecto ya sirve para un fraude telefonico.')
print('El umbral que importa no es "indistinguible", es "suficiente para enganar 20 segundos".')
print('Ese umbral se cruzo antes que el de calidad.')

## 12. Corrección

Cuánta identidad cabe en cada duración:


In [ ]:
r = run_paper_lab('vall_e', seed=3)['result']
for f in r['identificacion_por_duracion']:
    print(f)
print('enfoques:', r['comparacion_de_enfoques'])

## 13. Desafío guiado

Explica por qué la saturación de la curva es lo que convierte esto en un problema de política y no solo de ingeniería.


In [ ]:
r = run_paper_lab('vall_e', seed=3)['result']
show(r)

## 14. Desafío autónomo

Revisa qué política de consentimiento aplicarías si tuvieras que desplegar clonación de voz: qué prueba de autorización pedirías y cómo la verificarías.


## 15. Evidencia de aprendizaje

Guarda tu política, con el mecanismo concreto de verificación.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P130_vall_e/README.md) · evaluación formal: [`assessments/papers/P130_vall_e.md`](../../assessments/papers/P130_vall_e.md)


## 16. Cierre

Si cualquier voz se puede copiar, hace falta saber qué se generó. Esa es P131.


## 17. Conexión con el siguiente hito

- P131

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
